# **Redes Neuronales Convolucionales**
### Actividad grupal

---

En esta actividad se trabajará con Redes Neuronales Convolucionales (Convolutional Neural Networks, CNN) para resolver un problema de clasificación de imágenes. En particular, se utilizará un conjunto de imágenes de mascotas, enfocándose en la clasificación de perros y gatos.

Dado que las CNN profundas son modelos computacionalmente demandantes, se recomienda realizar la práctica en Google Colaboratory, aprovechando el soporte para unidades de procesamiento gráfico (GPU). En el siguiente enlace se describe el procedimiento para habilitar una GPU en Colab:  
[Guía para activar GPU en Google Colab](https://medium.com/deep-learning-turkey/google-colab-free-gpu-tutorial-e113627b9f5d).

El conjunto de datos a utilizar es el **Oxford-IIIT Pet Dataset**, el cual contiene imágenes de distintas razas de perros y gatos. Su sitio oficial es:  
[Oxford-IIIT Pet Dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/)

Este dataset es considerablemente más complejo que otros utilizados previamente, como Fashion MNIST, ya que:
- Incluye múltiples clases (razas),
- Presenta variaciones significativas en escala, pose e iluminación,
- Contiene imágenes con diferentes fondos y encuadres.

El conjunto de datos puede integrarse fácilmente a flujos de trabajo basados en frameworks de aprendizaje profundo como [Tensorflow](https://www.tensorflow.org/datasets/catalog/oxford_iiit_pet) y [PyTorch](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.OxfordIIITPet.html), ya sea mediante utilidades propias de cada framework o mediante descarga directa desde el sitio oficial.

Antes de iniciar el desarrollo del modelo, se recomienda descargar el dataset, explorar visualmente las imágenes y analizar su estructura, con el fin de comprender mejor los retos asociados al problema de clasificación.

## Librerías

In [2]:
import cv2
import numpy as np
from tensorflow import keras
import matplotlib.pyplot as plt
# Libreria para el dataset
import tensorflow_datasets as tfds

In [3]:
from keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from keras.models import Sequential
from keras.utils import to_categorical
from keras.callbacks import ModelCheckpoint
from keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import time
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

## Dataset

El dataset se encuentra la librería ***tfds***. En caso de no tener la líbrería instalarla mediante
```
# !pip install tensorflow-datasets
```
Las imágenes se descargan a través del método *load*, que tiene los siguientes parámetros:
* **split** indica la división del dataset, por ejemplo: "train" o ["train", "test"], ...
* **data_dir** ubicación donde se almacenan los datos, por dafault es ~/
* **with_info** devuelve los metadatos del dataset (tfds.core.DatasetInfo). Por default es *True*
* **download** deshabilita la descarga del dataset. Por default es False
* **as_supervised** descarga el dataset con su salida, variable y.

In [4]:
split_data, info = tfds.load(
    'oxford_iiit_pet',
    split=[
        'train[:80]',     # entrenamiento
        'train[80%:90%]', # validación
        'train[90%:]'],   # prueba
    with_info = True,
    as_supervised=True
)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.GEP8X9_4.0.0/oxford_iiit_pet-train.tfrecord*...…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.GEP8X9_4.0.0/oxford_iiit_pet-test.tfrecord*...:…

Dataset oxford_iiit_pet downloaded and prepared to /root/tensorflow_datasets/oxford_iiit_pet/4.0.0. Subsequent calls will reuse this data.


In [5]:
# Las imágenes se estandarizan al tamaño 160x160
IMG_SIZE = 160

El manejo de los datos del conjunto se realiza mediante la **API tf.data**, una herramienta de *TensorFlow* diseñada para construir *pipelines* de entrada de datos eficientes y escalables. Esta API es especialmente útil cuando se trabaja con conjuntos de datos de gran tamaño, ya que permite cargar y procesar la información en lotes pequeños (*batches*), optimizando el uso de recursos y mejorando el rendimiento durante el entrenamiento.

Cada conjunto de datos en **tf.data** se representa como un objeto de tipo `tf.data.Dataset`, el cual puede contener elementos en forma de tuplas o diccionarios de tensores (por ejemplo, `tf.float32`, `tf.int32`, `tf.string`). Esta estructura garantiza compatibilidad total con *TensorFlow* y facilita un procesamiento eficiente, ya que los datos se consumen de manera incremental durante el entrenamiento, reduciendo la necesidad de cargar todo el dataset en memoria al mismo tiempo.

### **Operaciones comunes en tf.data**

Las operaciones de `tf.data` pueden encadenarse para construir *pipelines* de procesamiento flexibles y eficientes. Algunas de las más utilizadas son:

* **map**: Aplica una función personalizada a cada elemento del `Dataset` (por ejemplo, normalización de imágenes o aumento de datos).
* **batch**: Agrupa los elementos en lotes de un tamaño especificado, lo que permite el entrenamiento por mini-lotes.
* **shuffle**: Mezcla aleatoriamente los elementos del `Dataset` utilizando un parámetro `buffer_size`. Un valor grande favorece una mejor aleatorización, especialmente en conjuntos de datos extensos.
* **repeat**: Repite el conjunto de datos un número determinado de veces, útil para controlar el número de épocas durante el entrenamiento.
* **prefetch**: Permite preparar los siguientes lotes mientras el modelo se encuentra entrenando, mejorando la utilización de recursos de CPU y GPU.

Para profundizar en el uso y configuración de estos métodos, se recomienda consultar la documentación oficial de [tf.data](https://www.tensorflow.org/guide/data).

In [6]:
(train_ds, val_ds, test_ds) = split_data

## Ejercicio

Utilizando ***Convolutional Neural Networks*** (CNN) implementadas con Keras, entrenar un clasificador capaz de reconocer razas de perros y gatos en imágenes, alcanzando una accuracy en el conjunto de **test** de al menos **85%**. Es indispensable que el **mejor modelo seleccionado no presente evidencias de overfitting ni de underfitting**.

El desarrollo de la actividad debe apoyarse en un **proceso experimental riguroso**, documentado mediante un **informe técnico**, en el cual se analicen de manera comparativa las distintas estrategias exploradas y se justifiquen las decisiones adoptadas con base en los resultados obtenidos.

A continuación, se detallan los aspectos que deben ser abordados en el informe:
* **Análisis exploratorio de los datos utilizados**, considerando la distribución de clases, el tamaño del dataset, ejemplos representativos de las imágenes y la identificación de posibles desbalances.
* **Diseño y evaluación de al menos cuatro modelos**. Se espera que al menos uno de ellos corresponda a una arquitectura propuesta por los estudiantes. Adicionalmente, pueden emplearse modelos preentrenados mediante *transfer learning* o *fine-tuning*.
* **Análisis de resultados**, incluyendo métricas de *precision* y *recall* por clase y/o el uso de una matriz de confusión, con el fin de identificar qué clases presentan mejor o peor desempeño.
* **Análisis visual de los errores del modelo**, discutiendo qué tipos de imágenes o qué razas generan mayor confusión.
* **Comparación del desempeño de modelos basados en CNN frente a un modelo Fully Connected** (quinto modelo), considerando las imágenes aplanadas como entrada.
* **Entrenamiento y comparación de distintas arquitecturas CNN**, discutiendo aspectos como la profundidad de la red, hiperparámetros, optimizador, funciones de activación, uso de técnicas de regularización y batch normalization, entre otros.
* **Empleo de técnicas de data augmentation** y análisis de su impacto en el desempeño y la capacidad de generalización del modelo.

---
### **Notas importantes**

* Los conjuntos de **training** y **validation** deben emplearse durante el entrenamiento y ajuste de los modelos. El conjunto de **test** debe utilizarse **exclusivamente para la evaluación final del desempeño**.
* Evitar el reentrenamiento de los modelos, salvar el mejor modelo de cada modelo empleado, para que en caso de ser necesario se realicen las predicciones sin necesidad de emplear GPU.
* Es obligatorio verificar que el modelo no presente **overfitting**, apoyándose en el análisis de las curvas de entrenamiento y validación.
* No es necesario mostrar en el notebook las trazas de entrenamiento de todos los modelos evaluados. No obstante, se recomienda conservar las gráficas de entrenamiento para sustentar el análisis en el informe. **De manera obligatoria, debe mostrarse el entrenamiento completo y la evaluación sobre el conjunto de test del mejor modelo obtenido**.
* Las imágenes **no se encuentran normalizadas**, por lo que es necesario aplicar un proceso de normalización previo al entrenamiento del modelo.

---
### **Notas adicionales**

* Con el fin de **optimizar el uso de recursos computacionales**, se debe **evitar el reentrenamiento innecesario de los modelos**. Para cada arquitectura evaluada, es obligatorio **guardar el mejor modelo obtenido**, utilizando como criterio su desempeño sobre el conjunto de validación.
* En caso de requerir nuevas predicciones, análisis de errores o comparaciones adicionales, estas deberán realizarse **cargando los modelos previamente almacenados**, sin necesidad de volver a entrenarlos ni de emplear GPU.
* Para garantizar la **reproducibilidad de los experimentos**, se recomienda fijar de manera consistente las semillas aleatorias (*random seeds*) asociadas al entrenamiento, así como documentar los hiperparámetros, configuraciones del optimizador y criterios de parada utilizados en cada modelo.
* La **trazabilidad experimental** debe quedar reflejada en el notebook y en el informe técnico, de modo que sea posible identificar claramente qué configuraciones dieron lugar a cada resultado reportado. Esto incluye la asociación explícita entre modelos entrenados, métricas obtenidas, curvas de entrenamiento y conclusiones derivadas.